# Week 10 – Strategy Comparison

This notebook compares three strategies on AAPL data over a historical window:
1. Buy‑and‑hold
2. SMA crossover (50‑day vs 200‑day)
3. LLM‑guided strategy (using the Week 8 simulation)

It computes performance metrics (Sharpe ratio, max drawdown, win rate, total return) using `src/evaluation/metrics.py` and generates a comparison table and equity‑curve plot.
Finally, it writes a markdown report to `reports/evaluation_report.md`.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
sys.path.append('.')  # allow import from src
from src.evaluation.metrics import compute_sharpe, compute_max_drawdown, compute_win_rate
from zipline.api import order_target, symbol, set_slippage, set_commission, slippage, commission
from zipline import run_algorithm
from zipline.pipeline import USEquityPricing
from zipline.pipeline.factors import SimpleMovingAverage
from zipline.filters import static
from src.simulation.orchestrator import simulate_historical


In [ ]:
# Configuration
ticker = "AAPL"
start_date = "2023-01-01"
end_date   = "2023-12-31"
starting_capital = 10_000.0

print(f"Running comparison for {ticker} from {start_date} to {end_date}...")


In [ ]:
# Helper: compute buy‑and‑hold equity curve\ndef buy_and_hold(context, data):\n    if not hasattr(context, 'bought'):\n        order_target(context.asset, context.portfolio.portfolio_value / data.current(context.asset, 'price'))\n        context.bought = True\n

In [ ]:
# Helper: SMA crossover strategy\ndef sma_crossover_initialize(context):\n    context.sma_50 = SimpleMovingAverage(inputs=[USEquityPricing.close], window_length=50, mask=static())\n    context.sma_200 = SimpleMovingAverage(inputs=[USEquityPricing.close], window_length=200, mask=static())\n    context.asset = symbol(ticker)\n    set_slippage(slippage.FixedSlippage(spread=0))\n    set_commission(commission.PerShare(cost=0.005))\n

In [ ]:
def sma_crossover_handle_data(context, data):\n    if not data.can_trade(context.asset):\n        return\n    sma_50 = context.sma_50[data.date][context.asset]\n    sma_200 = context.sma_200[data.date][context.asset]\n    if np.isnan(sma_50) or np.isnan(sma_200):\n        return\n    if sma_50 > sma_200:\n        target = 1.0  # fully invested\n    else:\n        target = 0.0  # out of market\n    order_target(context.asset, target * context.portfolio.portfolio_value / data.current(context.asset, 'price'))\n

In [ ]:
# Run baseline strategies\nprint('\nRunning baseline strategies...')\n# Buy‑and‑hold\nresult_bh = run_algorithm(\n    start=pd.Timestamp(start_date, tz='utc'),\n    end=pd.Timestamp(end_date, tz='utc'),\n    initialize=lambda c: setattr(c, 'asset', symbol(ticker)) or setattr(c, 'bought', False),\n    handle_data=buy_and_hold,\n    capital_base=starting_capital,\n    bundle='raats_daily'\n)\nequity_bh = result_bh.portfolio_value\nreturns_bh = result_bh.returns\n
# SMA crossover\nresult_sma = run_algorithm(\n    start=pd.Timestamp(start_date, tz='utc'),\n    end=pd.Timestamp(end_date, tz='utc'),\n    initialize=sma_crossover_initialize,\n    handle_data=sma_crossover_handle_data,\n    capital_base=starting_capital,\n    bundle='raats_daily'\n)\nequity_sma = result_sma.portfolio_value\nreturns_sma = result_sma.returns\n

In [ ]:
# Run LLM‑guided strategy via historical simulation\nprint('\nRunning LLM‑guided simulation...')\nllm_result = simulate_historical(\n    ticker=ticker,\n    start_date=start_date,\n    end_date=end_date,\n    portfolio_value=starting_capital,\n    verbose=False\n)\nclosed_trades = llm_result.get('closed_trades', [])\nif closed_trades:\n    pnl_list = [t.get('pnl', 0.0) for t in closed_trades]\n    equity_llm = [starting_capital]\n    for pnl in pnl_list:\n        equity_llm.append(equity_llm[-1] + pnl)\n    equity_llm = np.array(equity_llm)\n    returns_llm = np.diff(equity_llm) / equity_llm[:-1]\nelse:\n    equity_llm = np.array([starting_capital])\n    returns_llm = np.array([])\n

In [ ]:
# Compute metrics for each strategy\ndef compute_strategy_metrics(returns, equity, starting_capital):\n    if len(returns) == 0:\n        return {\n            'sharpe': None,\n            'max_dd': None,\n            'win_rate': None,\n            'total_return': 0.0,\n            'final_equity': starting_capital\n        }\n    sharpe = compute_sharpe(returns)\n    max_dd = compute_max_drawdown(equity)\n    # For win‑rate we need trade P&L; we approximate from the equity curve\n    # by counting periods where returns > 0 as winning days (not perfect).\n    # Better to use closed trades for LLM, but for consistency we'll use daily returns>0.\n    win_rate = (returns > 0).sum() / len(returns) if len(returns) > 0 else 0.0\n    total_return = (equity[-1] / starting_capital - 1)\n    return {\n        'sharpe': sharpe,\n        'max_dd': max_dd,\n        'win_rate': win_rate,\n        'total_return': total_return,\n        'final_equity': equity[-1]\n    }\n
metrics_bh = compute_strategy_metrics(returns_bh, equity_bh, starting_capital)\nmetrics_sma = compute_strategy_metrics(returns_sma, equity_sma, starting_capital)\nmetrics_llm = compute_strategy_metrics(returns_llm, equity_llm, starting_capital)\n

In [ ]:
# Print comparison table\nprint('\n=== Strategy Comparison ===')\nprint(f'{'Strategy':<20} {'Sharpe':>10} {'MaxDD':>10} {'WinRate':>10} {'TotalRet':>10}')\nprint('-'*70)\nfor name, m in [('Buy‑and‑hold', metrics_bh), ('SMA 50/200', metrics_sma), ('LLM‑guided', metrics_llm)]:\n    sharpe = f'{m["sharpe"]:.3f}' if m['sharpe'] is not None else 'None'\n    max_dd = f'{m["max_dd"]:.2%}' if m['max_dd'] is not None else 'None'\n    win_rate = f'{m["win_rate"]:.2%}' if m['win_rate'] is not None else 'None'\n    total_ret = f'{m["total_return"]:.2%}'\n    print(f'{name:<20} {sharpe:>10} {max_dd:>10} {win_rate:>10} {total_ret:>10}')\n

In [ ]:
# Plot equity curves\nplt.figure(figsize=(12,6))\nplt.plot(equity_bh.index, equity_bh.values, label='Buy‑and‑hold', linewidth=2)\nplt.plot(equity_sma.index, equity_sma.values, label='SMA 50/200', linewidth=2)\nif len(equity_llm) > 1:\n    # equity_llm is indexed by trade count, not dates; we need to align to dates\n    # For simplicity, we'll plot against trade index, but note the difference.\n    # Better: we could create a date index by assigning each trade to a date, but we skip for now.\n    plt.plot(np.arange(len(equity_llm)), equity_llm, label='LLM‑guided (trade #)', linewidth=2, marker='o', markevery=max(1,len(equity_llm)//10))\nplt.title(f'Equity Curves – Strategy Comparison ({ticker})')\nplt.ylabel('Portfolio Value ($)')\nplt.xlabel('Date (for BH and SMA) / Trade # (for LLM)')\nplt.legend()\nplt.grid(True)\nplt.show()


In [ ]:
# Write markdown report\nreport_dir = 'reports'
os.makedirs(report_dir, exist_ok=True)\nreport_path = os.path.join(report_dir, 'evaluation_report.md')\n
with open(report_path, 'w', encoding='utf-8') as f:\n    f.write('# Evaluation Report\n\n')\n    f.write(f'**Period:** {start_date} to {end_date}  \n')\n    f.write(f'**Ticker:** {ticker}  \n')\n    f.write(f'**Starting Capital:** ${starting_capital:,.2f}  \n\n')\n    f.write('## Performance Metrics\n\n')\n    f.write('| Strategy | Sharpe Ratio | Max Drawdown | Win Rate | Total Return | Final Equity |\n')\n    f.write('|----------|--------------|--------------|----------|--------------|--------------|\n')\n    for name, m in [('Buy‑and‑hold', metrics_bh), ('SMA 50/200', metrics_sma), ('LLM‑guided', metrics_llm)]:\n        sharpe = f'{m["sharpe"]:.3f}' if m['sharpe'] is not None else 'N/A'\n        max_dd = f'{m["max_dd"]:.2%}' if m['max_dd'] is not None else 'N/A'\n        win_rate = f'{m["win_rate"]:.2%}' if m['win_rate'] is not None else 'N/A'\n        total_ret = f'{m["total_return"]:.2%}'\n        final_eq = f'${m["final_equity"]:,.2f}'\n        f.write(f'| {name} | {sharpe} | {max_dd} | {win_rate} | {total_ret} | {final_eq} |\n')\n    f.write('\n## Observations\n\n')\n    f.write('1. **Buy‑and‑hold** serves as a passive benchmark.\n')\n    f.write('2. **SMA 50/200** is a simple trend‑following rule.\n')\n    f.write('3. **LLM‑guided** strategy uses the agent\'s daily signal generation (news sentiment + LLM analysis) to decide entries and exits.\n')\n    f.write('\n## Conclusion\n\n')\n    f.write('Based on the simulated period, the LLM‑guided strategy showed [insert conclusion based on the metrics above]. Further analysis over multiple windows and tickers is recommended.\n')\n